In [30]:
import pandas as pd
import os

# Mapping of month names to numbers
month_mapping = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}

# Get the current script directory
script_dir = os.path.dirname(os.path.abspath("__file__"))

# Load the Excel file
file_path = os.path.join(script_dir, "./publications.xlsx")

# Define the output folder (_publications)
output_folder = os.path.join(script_dir, "../_publications")

# Ensure the _publications folder exists
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Read the Excel file
df = pd.read_excel(file_path)

# Convert Year to integer
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").fillna(0).astype(int)

# Convert Month names to numbers using mapping
df["Month"] = df["Month"].astype(str).str.strip().map(month_mapping).fillna(0).astype(int)

# Create a sorting key in the format YYYYMM (e.g., 202503 for March 2025)
df["sort_key"] = df["Year"] * 100 + df["Month"]

# **Sorting: First by Year (Descending), then by Month (Descending within each Year)**
df = df.sort_values(by=["sort_key"], ascending=[False])

# Iterate over each row and create a Markdown (.md) file
for index, row in df.iterrows():
    citation = str(row["Citation"]).strip()
    category = str(row["Category"]).strip()
    paper_link = str(row["Paper Link"]).strip()
    year = str(row["Year"]).strip()
    month = int(row["Month"])
    sort_key = str(row["sort_key"])  # ✅ Store sorting key in .md file

    # Fix title formatting
    citation = citation.replace("‘", '"').replace("’", '"')
    citation = citation.replace("“", '"').replace("”", '"')
    citation = citation.replace('"', "“", 1).replace('"', "”", 1)

    # Make "M. Z. Islam" bold
    citation = citation.replace("M. Z. Islam", "**M. Z. Islam**")

    # Generate a safe filename
    safe_title = "".join(c if c.isalnum() or c in (" ", "-") else "" for c in citation[:50]).replace(" ", "-").lower()
    md_filename = f"{year}-{month:02d}-{safe_title}.md"

    # Construct Markdown content with `sort_key`
    md_content = f"""---
title: "{citation}"
year: "{year}"
month: "{month}"
sort_key: "{sort_key}"
category: "{category}"
paper_link: "{paper_link}"
# layout: publication
---
**Citation:**  
{citation}

**Full Paper:**  
[Read Here]({paper_link})
"""

    # Write the Markdown file
    with open(os.path.join(output_folder, md_filename), "w", encoding="utf-8") as f:
        f.write(md_content)

print(f"Markdown files successfully saved in {output_folder}/")


Markdown files successfully saved in /Users/zahidulislam/zahidul-ece.github.io/py_publication_n/../_publications/
